# Anticipatory Aerodynamic Mode Prediction — Kaggle Training Pipeline
**Paper**: *Anticipatory Aerodynamic Mode Prediction in Formula 1* — IEEE  
**Circuits**: Monza · Monaco · Silverstone · Suzuka (LOCO evaluation)  
**Models**: LR-instant, RF-instant, RF-lag, CNN, LSTM, GRU, TCN, CausalTransformer  
**Checkpoints**: saved every 10 epochs — interrupt and resume safely  
> Attach **Research Datasets** to this notebook before running.


In [12]:
# Cell 1 — GPU check
import subprocess, sys
import torch

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU     : {p.name}")
    print(f"VRAM    : {p.total_memory / 1e9:.1f} GB")
    print(f"SM count: {p.multi_processor_count}")
    USE_AMP = True
else:
    print("WARNING: No GPU — CPU fallback active (slow but functional).")
    USE_AMP = False
print(f"AMP     : {USE_AMP}")


PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
VRAM    : 15.6 GB
SM count: 40
AMP     : True


In [13]:
# Cell 2 — Imports & configuration
import copy, json, pickle, random, traceback, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, f1_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# ── Paths ──────────────────────────────────────────────────────────────────────
INPUT_ROOT  = Path("/kaggle/input")
WORK_DIR    = Path("/kaggle/working")
CKPT_DIR    = WORK_DIR / "checkpoints"
RESULTS_DIR = WORK_DIR / "results"
MODELS_DIR  = WORK_DIR / "models"
for d in [CKPT_DIR, RESULTS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Dataset constants ──────────────────────────────────────────────────────────
W               = 50        # window length (~5 s at 10 Hz)
HORIZONS        = [1, 5, 10, 25, 50]
SPEED_THRESHOLD = 240.0     # km/h — X-mode activation
GEAR_THRESHOLD  = 6
MASS_KG         = 798.0     # F1 car minimum (2026 regs)
SEED            = 42

FEATURE_COLS = [
    "Speed", "RPM", "nGear", "Throttle", "Brake",
    "X", "Y", "Z",
    "Acceleration", "Elevation_Delta",
    "Kinetic_Energy_MJ", "Longitudinal_Force_N",
]
F_DIM = len(FEATURE_COLS)   # 12

# Candidate filenames per circuit (checked in order)
CSV_MAP = {
    "Monza":       ["monza_red_bull.csv",       "monza.csv"],
    "Monaco":      ["monaco_red_bull.csv",      "monaco.csv"],
    "Silverstone": ["silverstone_red_bull.csv", "silverstone.csv"],
    "Suzuka":      ["suzuka_red_bull.csv",       "suzuka.csv"],
}

# ── Training hyperparameters ───────────────────────────────────────────────────
BATCH_SIZE  = 512   # P100/T4 can handle 512 comfortably
MAX_EPOCHS  = 100
PATIENCE    = 15
LR          = 3e-4
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP     = torch.cuda.is_available()

print(f"Device  : {DEVICE}")
print(f"AMP     : {USE_AMP}")
print(f"Features ({F_DIM}): {FEATURE_COLS}")
print(f"W={W}  Horizons={HORIZONS}  Batch={BATCH_SIZE}")


Device  : cuda
AMP     : True
Features (12): ['Speed', 'RPM', 'nGear', 'Throttle', 'Brake', 'X', 'Y', 'Z', 'Acceleration', 'Elevation_Delta', 'Kinetic_Energy_MJ', 'Longitudinal_Force_N']
W=50  Horizons=[1, 5, 10, 25, 50]  Batch=512


In [14]:
# Cell 3 — Find & load CSVs from /kaggle/input
def find_csvs(csv_map):
    # Index all CSVs under input root
    all_csvs = {p.name.lower(): p for p in INPUT_ROOT.rglob("*.csv")}
    print(f"All CSVs found under {INPUT_ROOT}:")
    for name in sorted(all_csvs):
        print(f"  {all_csvs[name]}")

    found = {}
    for circuit, candidates in csv_map.items():
        for cand in candidates:
            if cand.lower() in all_csvs:
                found[circuit] = all_csvs[cand.lower()]
                break
        if circuit not in found:
            # fuzzy fallback: any file whose name contains the circuit name
            for name, path in all_csvs.items():
                if circuit.lower() in name:
                    found[circuit] = path
                    break
        status = f"→ {found[circuit]}" if circuit in found else "NOT FOUND"
        print(f"[{'OK' if circuit in found else 'MISS'}] {circuit:12s}  {status}")

    return found

csv_paths = find_csvs(CSV_MAP)
print(f"\nMatched {len(csv_paths)}/{len(CSV_MAP)} circuits")
if len(csv_paths) < 2:
    raise RuntimeError(
        "Need ≥2 circuit CSVs for LOCO. "
        "Make sure the Research Datasets dataset is attached to this notebook.")


All CSVs found under /kaggle/input:
  /kaggle/input/datasets/anasnorani/research-datasets/monaco_red_bull.csv
  /kaggle/input/datasets/anasnorani/research-datasets/monza_red_bull.csv
  /kaggle/input/datasets/anasnorani/research-datasets/silverstone_red_bull.csv
  /kaggle/input/datasets/anasnorani/research-datasets/suzuka_red_bull.csv
[OK] Monza         → /kaggle/input/datasets/anasnorani/research-datasets/monza_red_bull.csv
[OK] Monaco        → /kaggle/input/datasets/anasnorani/research-datasets/monaco_red_bull.csv
[OK] Silverstone   → /kaggle/input/datasets/anasnorani/research-datasets/silverstone_red_bull.csv
[OK] Suzuka        → /kaggle/input/datasets/anasnorani/research-datasets/suzuka_red_bull.csv

Matched 4/4 circuits


In [15]:
# Cell 4 — Feature engineering & CSV loading
def _coerce_brake(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(np.float32)
    if series.dtype == object:
        return (series.map({"True":1.,"False":0.,"true":1.,"false":0.,"1":1.,"0":0.})
                      .fillna(0.).astype(np.float32))
    return pd.to_numeric(series, errors="coerce").fillna(0.).astype(np.float32)


def load_and_clean(csv_path, circuit):
    df = pd.read_csv(csv_path)
    df["Circuit"] = circuit
    for col in ["Speed","RPM","nGear","Throttle","X","Y","Z",
                "LapNumber","Time_Elapsed_Sec"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df["Brake"] = _coerce_brake(df["Brake"]) if "Brake" in df.columns else 0.0
    df = (df.sort_values(["Driver","LapNumber","Time_Elapsed_Sec"], kind="mergesort")
            .reset_index(drop=True))

    # Derived features (compute only if absent)
    for feat, base in [("Acceleration","Speed"), ("Elevation_Delta","Z")]:
        if feat not in df.columns:
            df[feat] = (df.groupby(["Driver","LapNumber"])[base]
                          .diff().fillna(0.))
        else:
            df[feat] = pd.to_numeric(df[feat], errors="coerce").fillna(0.)

    speed = df["Speed"].fillna(0.)
    accel = df["Acceleration"].fillna(0.)
    df["Kinetic_Energy_MJ"]    = 0.5 * MASS_KG * (speed / 3.6)**2 / 1e6
    df["Longitudinal_Force_N"] = MASS_KG * (accel / 3.6)

    df["LapKey"] = (circuit + "|" + df["Driver"].astype(str)
                    + "|L" + df["LapNumber"].fillna(-1).astype(int).astype(str))

    before = len(df)
    df = df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
    if before - len(df) > 0:
        warnings.warn(f"{circuit}: dropped {before - len(df)} NaN rows")
    return df


raw_dfs = {}
for circuit, path in csv_paths.items():
    try:
        df = load_and_clean(path, circuit)
        raw_dfs[circuit] = df
        xm = ((df["Speed"] >= SPEED_THRESHOLD) & (df["nGear"] >= GEAR_THRESHOLD)).mean()
        print(f"[OK] {circuit:12s}  rows={len(df):7,}  laps={df['LapKey'].nunique():3d}"
              f"  drivers={df['Driver'].nunique()}  inst_X-mode={xm:.3f}")
    except Exception:
        print(f"[ERROR] loading {circuit}:")
        traceback.print_exc()

print(f"\n{len(raw_dfs)} circuits ready.")


[OK] Monza         rows= 31,098  laps= 50  drivers=1  inst_X-mode=0.603
[OK] Monaco        rows= 39,881  laps= 70  drivers=1  inst_X-mode=0.142
[OK] Silverstone   rows= 25,060  laps= 33  drivers=1  inst_X-mode=0.423
[OK] Suzuka        rows= 63,673  laps= 89  drivers=2  inst_X-mode=0.390

4 circuits ready.


In [16]:
# Cell 5 — Diagnostics
for circuit, df in raw_dfs.items():
    present = [c for c in FEATURE_COLS if c in df.columns]
    missing = [c for c in FEATURE_COLS if c not in df.columns]
    print(f"{'='*56}")
    print(f"Circuit : {circuit}")
    print(f"Shape   : {df.shape}")
    print(f"Features: {len(present)}/{F_DIM} present" +
          (f"  ← MISSING: {missing}" if missing else " ✓"))
    print(f"NaNs    : {df[FEATURE_COLS].isnull().sum().sum()}")
    print(f"Laps    : {df['LapKey'].nunique()}")
    for col in ["Speed", "RPM", "nGear", "Throttle"]:
        if col in df.columns:
            s = df[col]
            print(f"  {col:22s}  min={s.min():.1f}  max={s.max():.1f}"
                  f"  mean={s.mean():.1f}  nan={s.isna().sum()}")
print("\nDiagnostics complete.")


Circuit : Monza
Shape   : (31098, 40)
Features: 12/12 present ✓
NaNs    : 0
Laps    : 50
  Speed                   min=58.0  max=352.0  mean=251.0  nan=0
  RPM                     min=5272.0  max=12440.0  mean=10690.2  nan=0
  nGear                   min=1.0  max=8.0  mean=6.3  nan=0
  Throttle                min=0.0  max=100.0  mean=78.2  nan=0
Circuit : Monaco
Shape   : (39881, 40)
Features: 12/12 present ✓
NaNs    : 0
Laps    : 70
  Speed                   min=43.0  max=288.0  mean=156.1  nan=0
  RPM                     min=4503.0  max=12633.0  mean=9443.6  nan=0
  nGear                   min=1.0  max=7.0  mean=3.8  nan=0
  Throttle                min=0.0  max=100.0  mean=48.1  nan=0
Circuit : Silverstone
Shape   : (25060, 40)
Features: 12/12 present ✓
NaNs    : 0
Laps    : 33
  Speed                   min=62.0  max=332.0  mean=208.2  nan=0
  RPM                     min=5387.0  max=12478.0  mean=10181.1  nan=0
  nGear                   min=2.0  max=8.0  mean=5.1  nan=0
  Throttle   

In [17]:
# Cell 6 — Windowing + LOCO scalers (all horizons)
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)

set_seed()


def build_windows(dfs_dict, W, H):
    all_X, all_y, all_c, all_k = [], [], [], []
    for circuit, df in dfs_dict.items():
        for lap_key, lap_df in df.groupby("LapKey", sort=False):
            lap_df = lap_df.sort_values("Time_Elapsed_Sec").reset_index(drop=True)
            L = len(lap_df)
            if L < W + H:
                continue
            feat  = lap_df[FEATURE_COLS].to_numpy(dtype=np.float32)
            speed = lap_df["Speed"].to_numpy(dtype=np.float32)
            gear  = lap_df["nGear"].to_numpy(dtype=np.float32)
            for i in range(L - W - H + 1):
                label_idx = i + W + H - 1      # strictly outside input [i..i+W-1]
                all_X.append(feat[i:i + W])
                all_y.append(int(speed[label_idx] >= SPEED_THRESHOLD
                                 and gear[label_idx] >= GEAR_THRESHOLD))
                all_c.append(circuit)
                all_k.append(str(lap_key))
    if not all_X:
        raise ValueError(f"No windows for W={W}, H={H}.")
    return (np.stack(all_X),
            np.array(all_y, dtype=np.int8),
            np.array(all_c, dtype=object),
            np.array(all_k, dtype=object))


def fit_loco_scalers(X, circuits, circuit_list):
    N, Ww, Ff = X.shape
    return {
        held_out: StandardScaler().fit(X[circuits != held_out].reshape(-1, Ww * Ff))
        for held_out in circuit_list
    }


circuit_list  = sorted(raw_dfs.keys())
windows_cache = {}   # H → (X, y, circuits, lap_keys)
scalers_cache  = {}  # H → {held_out → scaler}

print("Building windows for all horizons …")
for H in HORIZONS:
    try:
        X, y, circs, lkeys = build_windows(raw_dfs, W, H)
        windows_cache[H]   = (X, y, circs, lkeys)
        scalers_cache[H]   = fit_loco_scalers(X, circs, circuit_list)
        print(f"  H={H:3d}  windows={len(y):8,}  X-mode={y.mean():.3f}  shape={X.shape}")
    except Exception:
        print(f"  [ERROR] H={H}:"); traceback.print_exc()

print("\nWindowing complete.")


Building windows for all horizons …
  H=  1  windows= 147,612  X-mode=0.344  shape=(147612, 50, 12)
  H=  5  windows= 146,644  X-mode=0.342  shape=(146644, 50, 12)
  H= 10  windows= 145,434  X-mode=0.339  shape=(145434, 50, 12)
  H= 25  windows= 141,804  X-mode=0.332  shape=(141804, 50, 12)
  H= 50  windows= 135,754  X-mode=0.339  shape=(135754, 50, 12)

Windowing complete.


In [18]:
# Cell 7 — Baseline models (LR-instant, RF-instant, RF-lag)
def compute_metrics(y_true, y_prob, y_pred):
    n_pos, n_neg = int(y_true.sum()), int((1 - y_true).sum())
    return {
        "n_test"    : len(y_true),
        "xmode_rate": float(y_true.mean()),
        "f1_macro"  : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_xmode"  : float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "f1_zmode"  : float(f1_score(y_true, y_pred, pos_label=0, zero_division=0)),
        "auc_roc"   : float(roc_auc_score(y_true, y_prob))
                      if n_pos > 0 and n_neg > 0 else float("nan"),
        "auc_pr"    : float(average_precision_score(y_true, y_prob))
                      if n_pos > 0 else float("nan"),
    }


baseline_rows = []

for H in HORIZONS:
    if H not in windows_cache:
        continue
    X, y, circs, _ = windows_cache[H]
    N, Ww, Ff = X.shape
    print(f"\n── Baselines H={H} ──────────────────────────────────────────────")

    for held_out in circuit_list:
        te_m   = circs == held_out
        tr_m   = ~te_m
        scaler = scalers_cache[H][held_out]

        X_tr_flat = X[tr_m].reshape(-1, Ww * Ff)
        X_te_flat = X[te_m].reshape(-1, Ww * Ff)
        y_tr = y[tr_m]; y_te = y[te_m]

        X_tr_s = scaler.transform(X_tr_flat)
        X_te_s = scaler.transform(X_te_flat)

        # instant: last time-step only
        X_tr_inst = X[tr_m][:, -1, :]
        X_te_inst = X[te_m][:, -1, :]
        inst_sc = StandardScaler().fit(X_tr_inst)
        X_tr_inst_s = inst_sc.transform(X_tr_inst)
        X_te_inst_s = inst_sc.transform(X_te_inst)

        for mname, clf, Xtr, Xte in [
            ("LR-instant",
             LogisticRegression(max_iter=500, C=1.0, class_weight="balanced",
                                random_state=SEED, solver="lbfgs"),
             X_tr_inst_s, X_te_inst_s),
            ("RF-instant",
             RandomForestClassifier(n_estimators=200, max_depth=12,
                                    class_weight="balanced", n_jobs=-1,
                                    random_state=SEED),
             X_tr_inst_s, X_te_inst_s),
            ("RF-lag",
             RandomForestClassifier(n_estimators=200, max_depth=12,
                                    class_weight="balanced", n_jobs=-1,
                                    random_state=SEED),
             X_tr_s, X_te_s),
        ]:
            try:
                clf.fit(Xtr, y_tr)
                y_prob = (clf.predict_proba(Xte)[:, 1]
                          if hasattr(clf, "predict_proba")
                          else clf.decision_function(Xte))
                y_pred = clf.predict(Xte)
                m = compute_metrics(y_te, y_prob, y_pred)
                baseline_rows.append({"model": mname, "held_out": held_out, "H": H, **m})
                print(f"  {mname:12s}  fold={held_out:12s}  "
                      f"F1={m['f1_macro']:.3f}  "
                      f"AUC-ROC={m['auc_roc']:.3f}  "
                      f"AUC-PR={m['auc_pr']:.3f}")
            except Exception:
                print(f"  [ERROR] {mname} fold={held_out}:"); traceback.print_exc()

if baseline_rows:
    baseline_df = pd.DataFrame(baseline_rows)
    baseline_df.to_csv(RESULTS_DIR / "baseline_results.csv", index=False)
    print(f"\nBaseline results saved ({len(baseline_rows)} rows).")
    print("\nMean across LOCO folds:")
    print(baseline_df.groupby("model")[["f1_macro", "auc_roc", "auc_pr"]]
          .mean().round(4).to_string())



── Baselines H=1 ──────────────────────────────────────────────
  LR-instant    fold=Monaco        F1=0.987  AUC-ROC=1.000  AUC-PR=0.998
  RF-instant    fold=Monaco        F1=0.986  AUC-ROC=1.000  AUC-PR=0.998
  RF-lag        fold=Monaco        F1=0.982  AUC-ROC=1.000  AUC-PR=0.997
  LR-instant    fold=Monza         F1=0.982  AUC-ROC=0.999  AUC-PR=0.999
  RF-instant    fold=Monza         F1=0.990  AUC-ROC=0.999  AUC-PR=1.000
  RF-lag        fold=Monza         F1=0.983  AUC-ROC=0.999  AUC-PR=0.999
  LR-instant    fold=Silverstone   F1=0.990  AUC-ROC=1.000  AUC-PR=0.999
  RF-instant    fold=Silverstone   F1=0.992  AUC-ROC=1.000  AUC-PR=1.000
  RF-lag        fold=Silverstone   F1=0.982  AUC-ROC=0.999  AUC-PR=0.999
  LR-instant    fold=Suzuka        F1=0.986  AUC-ROC=0.999  AUC-PR=0.996
  RF-instant    fold=Suzuka        F1=0.988  AUC-ROC=0.999  AUC-PR=0.998
  RF-lag        fold=Suzuka        F1=0.982  AUC-ROC=0.998  AUC-PR=0.994

── Baselines H=5 ─────────────────────────────────────────

In [19]:
# Cell 8 — PyTorch model definitions (exact match with 03_deep_models.py)
class FocalLoss(nn.Module):
    """Binary focal loss (Lin et al. 2017)."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma

    def forward(self, logits, targets):
        bce     = F.binary_cross_entropy_with_logits(logits, targets.float(), reduction="none")
        p_t     = torch.exp(-bce)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        return (alpha_t * (1 - p_t) ** self.gamma * bce).mean()


class CausalConv1d(nn.Module):
    """Conv1d with left-only padding (strictly causal)."""
    def __init__(self, in_ch, out_ch, kernel, dilation=1):
        super().__init__()
        self._pad = (kernel - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel, dilation=dilation)
    def forward(self, x):
        return self.conv(F.pad(x, (self._pad, 0)))


class CNN1D(nn.Module):
    def __init__(self, W, F, channels=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(F, channels, 5),    nn.BatchNorm1d(channels),   nn.ReLU(), nn.Dropout(dropout),
            CausalConv1d(channels, channels*2, 5), nn.BatchNorm1d(channels*2), nn.ReLU(), nn.Dropout(dropout),
            CausalConv1d(channels*2, channels*2, 5), nn.BatchNorm1d(channels*2), nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(channels*2, channels), nn.ReLU(), nn.Dropout(dropout+0.1), nn.Linear(channels, 1))
    def forward(self, x):   # (B,W,F) → (B,)
        return self.head(self.net(x.transpose(1,2)).mean(dim=-1)).squeeze(-1)


class CausalLSTM(nn.Module):
    def __init__(self, F, hidden=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(F, hidden, num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.)
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Dropout(dropout+0.1), nn.Linear(hidden//2, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


class CausalGRU(nn.Module):
    def __init__(self, F, hidden=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(F, hidden, num_layers=num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.)
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Dropout(dropout+0.1), nn.Linear(hidden//2, 1))
    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out[:, -1, :]).squeeze(-1)


class _TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel, dilation, dropout):
        super().__init__()
        self.conv1 = CausalConv1d(in_ch, out_ch, kernel, dilation)
        self.conv2 = CausalConv1d(out_ch, out_ch, kernel, dilation)
        self.bn1   = nn.BatchNorm1d(out_ch); self.bn2 = nn.BatchNorm1d(out_ch)
        self.drop  = nn.Dropout(dropout)
        self.proj  = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None
    def forward(self, x):
        res = x if self.proj is None else self.proj(x)
        x   = self.drop(F.relu(self.bn1(self.conv1(x))))
        x   = self.drop(F.relu(self.bn2(self.conv2(x))))
        return F.relu(x + res)


class TCN(nn.Module):
    """Dilated causal TCN — receptive field ≥ 121 > W=50."""
    def __init__(self, F, channels=64, kernel=5, dropout=0.2):
        super().__init__()
        in_ch = F; layers = []
        for d in [1, 2, 4, 8]:
            layers.append(_TCNBlock(in_ch, channels, kernel, d, dropout)); in_ch = channels
        self.network = nn.Sequential(*layers)
        self.head = nn.Sequential(
            nn.Linear(channels, channels//2), nn.ReLU(), nn.Dropout(dropout+0.1), nn.Linear(channels//2, 1))
    def forward(self, x):
        return self.head(self.network(x.transpose(1,2))[:,:,-1]).squeeze(-1)


class _SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=200):
        super().__init__()
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class CausalTransformer(nn.Module):
    """Causal Transformer encoder — last-token classification."""
    def __init__(self, F, d_model=64, nhead=4, num_layers=2, dim_ff=128, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(F, d_model)
        self.pos_enc    = _SinusoidalPE(d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head    = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    @staticmethod
    def _causal_mask(seq_len, device):
        m = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
        return m.masked_fill(m == 1, float("-inf"))
    def forward(self, x):
        x = self.pos_enc(self.input_proj(x))
        x = self.encoder(x, mask=self._causal_mask(x.size(1), x.device), is_causal=True)
        return self.head(x[:, -1, :]).squeeze(-1)


def build_model(name, W, F):
    registry = {
        "CNN":         lambda: CNN1D(W, F),
        "LSTM":        lambda: CausalLSTM(F),
        "GRU":         lambda: CausalGRU(F),
        "TCN":         lambda: TCN(F),
        "Transformer": lambda: CausalTransformer(F),
    }
    if name not in registry:
        raise ValueError(f"Unknown model '{name}'. Choose from {list(registry)}")
    return registry[name]()


print("Model parameter counts:")
for name in ["CNN","LSTM","GRU","TCN","Transformer"]:
    m = build_model(name, W, F_DIM)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  {name:14s} {n:>9,} params")


Model parameter counts:
  CNN              136,001 params
  LSTM             213,121 params
  GRU              161,921 params
  TCN              151,681 params
  Transformer       67,969 params


In [20]:
# Cell 9 — Training infrastructure (AMP + checkpoint + early stopping)
PROGRESS_FILE = CKPT_DIR / "progress.json"


def progress_key(model_name, H, held_out):
    return f"{model_name}_H{H:03d}_{held_out}"


def load_progress():
    if PROGRESS_FILE.exists():
        return set(json.load(PROGRESS_FILE.open())["completed"])
    return set()


def save_progress(done):
    json.dump({"completed": list(done)}, PROGRESS_FILE.open("w"), indent=2)


def make_tensor_ds(X, y, scaler):
    N, Ww, Ff = X.shape
    Xf = scaler.transform(X.reshape(N, -1)).reshape(N, Ww, Ff).astype(np.float32)
    return TensorDataset(torch.from_numpy(Xf), torch.from_numpy(y.astype(np.float32)))


def fit_fold(model_name, model, train_ds, val_ds, criterion,
             held_out, H, batch_size=BATCH_SIZE,
             max_epochs=MAX_EPOCHS, patience=PATIENCE):
    """
    Train one LOCO fold with:
      - AMP (automatic mixed precision) when CUDA is available
      - Checkpoint every 10 epochs → /kaggle/working/checkpoints/partial_*
      - Resume automatically if partial checkpoint exists
    Returns (trained_model, stopped_epoch).
    """
    partial_ckpt = CKPT_DIR / f"partial_{model_name}_H{H:03d}_{held_out}.pt"

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=5, min_lr=1e-6)
    amp_scaler = torch.cuda.amp.GradScaler() if USE_AMP else None

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True, num_workers=0, pin_memory=USE_AMP)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size * 2,
                              shuffle=False, num_workers=0)

    start_epoch = 1
    best_auc_pr = 0.0
    best_state  = {}
    no_improve  = 0

    # ── Resume from partial checkpoint ──────────────────────────────────────
    if partial_ckpt.exists():
        try:
            ckpt = torch.load(partial_ckpt, map_location=DEVICE)
            model.load_state_dict(ckpt["model_state"])
            optimizer.load_state_dict(ckpt["opt_state"])
            scheduler.load_state_dict(ckpt["sched_state"])
            if amp_scaler and ckpt.get("amp_state"):
                amp_scaler.load_state_dict(ckpt["amp_state"])
            start_epoch = ckpt["epoch"] + 1
            best_auc_pr = ckpt["best_auc_pr"]
            best_state  = ckpt["best_state"]
            no_improve  = ckpt["no_improve"]
            print(f"        ↺ Resumed from epoch {start_epoch} "
                  f"(best AUC-PR={best_auc_pr:.4f})")
        except Exception:
            print("        [WARN] Corrupt checkpoint — restarting fold.")
            partial_ckpt.unlink(missing_ok=True)

    # ── Training loop ────────────────────────────────────────────────────────
    for epoch in range(start_epoch, max_epochs + 1):

        # train
        model.train()
        total_loss = 0.0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            if USE_AMP:
                with torch.cuda.amp.autocast():
                    loss = criterion(model(X_b), y_b)
                amp_scaler.scale(loss).backward()
                amp_scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                amp_scaler.step(optimizer)
                amp_scaler.update()
            else:
                loss = criterion(model(X_b), y_b)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item()

        # validate
        model.eval()
        probs, lbls = [], []
        with torch.no_grad():
            for X_b, y_b in val_loader:
                probs.append(torch.sigmoid(model(X_b.to(DEVICE))).cpu().numpy())
                lbls.append(y_b.numpy())
        val_prob = np.concatenate(probs)
        val_true = np.concatenate(lbls)
        n_pos    = int(val_true.sum())
        val_auc  = (average_precision_score(val_true, val_prob)
                    if 0 < n_pos < len(val_true) else 0.5)
        scheduler.step(val_auc)

        if val_auc > best_auc_pr + 1e-5:
            best_auc_pr = val_auc
            best_state  = copy.deepcopy(model.state_dict())
            no_improve  = 0
        else:
            no_improve += 1

        if epoch % 5 == 0:
            lr_cur = optimizer.param_groups[0]["lr"]
            avg_loss = total_loss / len(train_loader)
            print(f"        ep={epoch:3d}  loss={avg_loss:.4f}  "
                  f"val_AUC-PR={val_auc:.4f}  best={best_auc_pr:.4f}  "
                  f"no_imp={no_improve}  lr={lr_cur:.1e}")

        # ── Checkpoint every 10 epochs ───────────────────────────────────────
        if epoch % 10 == 0:
            torch.save({
                "epoch"      : epoch,
                "model_state": model.state_dict(),
                "opt_state"  : optimizer.state_dict(),
                "sched_state": scheduler.state_dict(),
                "amp_state"  : amp_scaler.state_dict() if amp_scaler else None,
                "best_auc_pr": best_auc_pr,
                "best_state" : best_state,
                "no_improve" : no_improve,
            }, partial_ckpt)

        if no_improve >= patience:
            break

    # load best weights; clean up partial checkpoint
    if best_state:
        model.load_state_dict(best_state)
    partial_ckpt.unlink(missing_ok=True)
    return model, epoch


@torch.no_grad()
def evaluate_model(model, test_loader):
    model.eval()
    probs, lbls = [], []
    for X_b, y_b in test_loader:
        probs.append(torch.sigmoid(model(X_b.to(DEVICE))).cpu().numpy())
        lbls.append(y_b.numpy())
    y_prob = np.concatenate(probs)
    y_true = np.concatenate(lbls)
    y_pred = (y_prob >= 0.5).astype(np.int32)
    return y_true, y_prob, y_pred


print("Training infrastructure ready.")
print(f"Progress file: {PROGRESS_FILE}")
completed = load_progress()
print(f"Already completed: {len(completed)} fold(s)")


Training infrastructure ready.
Progress file: /kaggle/working/checkpoints/progress.json
Already completed: 14 fold(s)


In [21]:
# Cell 10 — Primary training: H=10, all 5 models, LOCO
H_PRIMARY      = 10
MODELS_PRIMARY = ["CNN", "LSTM", "GRU", "TCN", "Transformer"]

deep_rows = []

if H_PRIMARY not in windows_cache:
    print(f"ERROR: H={H_PRIMARY} windows not built — check Cell 6.")
else:
    X, y, circs, _ = windows_cache[H_PRIMARY]
    N, Ww, Ff = X.shape
    print(f"H={H_PRIMARY}  windows={N:,}  X-mode={y.mean():.3f}  shape={X.shape}\n")

    for held_out in circuit_list:
        te_m = circs == held_out
        tr_m = ~te_m
        n_tr, n_te = int(tr_m.sum()), int(te_m.sum())
        scaler = scalers_cache[H_PRIMARY][held_out]

        tr_idx = np.where(tr_m)[0]
        try:
            idx_tr, idx_val = train_test_split(
                tr_idx, test_size=0.10, stratify=y[tr_m], random_state=SEED)
        except ValueError:
            idx_tr, idx_val = train_test_split(tr_idx, test_size=0.10, random_state=SEED)

        train_ds    = make_tensor_ds(X[idx_tr],    y[idx_tr],    scaler)
        val_ds      = make_tensor_ds(X[idx_val],   y[idx_val],   scaler)
        test_ds     = make_tensor_ds(X[te_m],      y[te_m],      scaler)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2,
                                 shuffle=False, num_workers=0)

        pos_rate    = float(y[idx_tr].mean())
        criterion   = FocalLoss(alpha=1. - pos_rate, gamma=2.).to(DEVICE)

        print(f"── Fold hold-out={held_out}  train={n_tr:,}  test={n_te:,}"
              f"  pos_rate={pos_rate:.3f} ──")

        for mname in MODELS_PRIMARY:
            pk = progress_key(mname, H_PRIMARY, held_out)
            if pk in completed:
                print(f"  [{mname}] SKIPPED (checkpoint complete)")
                continue

            print(f"  [{mname}] training …", flush=True)
            try:
                set_seed(SEED)
                model = build_model(mname, W, F_DIM).to(DEVICE)
                model, ep = fit_fold(mname, model, train_ds, val_ds,
                                     criterion, held_out, H_PRIMARY)

                y_true, y_prob, y_pred = evaluate_model(model, test_loader)
                m = compute_metrics(y_true, y_prob, y_pred)

                print(f"  [{mname}] DONE  ep={ep:3d}  "
                      f"F1={m['f1_macro']:.3f}  "
                      f"AUC-ROC={m['auc_roc']:.3f}  "
                      f"AUC-PR={m['auc_pr']:.3f}")

                deep_rows.append({"model": mname, "held_out": held_out, "H": H_PRIMARY,
                                  "n_train": n_tr, "stopped_epoch": ep, **m})

                torch.save({"model_state": model.state_dict(), "W": W, "F": F_DIM,
                            "H": H_PRIMARY, "held_out": held_out,
                            "epoch": ep, "metrics": m},
                           MODELS_DIR / f"{mname}_fold_{held_out}_H{H_PRIMARY:03d}.pt")

                completed.add(pk)
                save_progress(completed)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            except Exception:
                print(f"  [{mname}] ERROR:"); traceback.print_exc()
        print()

    if deep_rows:
        deep_df = pd.DataFrame(deep_rows)
        deep_df.to_csv(RESULTS_DIR / "deep_results.csv", index=False)
        print(f"deep_results.csv saved ({len(deep_rows)} rows).")
        print("\nMean across LOCO folds (H=10):")
        print(deep_df.groupby("model")[["f1_macro","auc_roc","auc_pr"]]
              .mean().round(4).sort_values("auc_roc", ascending=False).to_string())


H=10  windows=145,434  X-mode=0.339  shape=(145434, 50, 12)

── Fold hold-out=Monaco  train=109,683  test=35,751  pos_rate=0.405 ──
  [CNN] SKIPPED (checkpoint complete)
  [LSTM] SKIPPED (checkpoint complete)
  [GRU] SKIPPED (checkpoint complete)
  [TCN] SKIPPED (checkpoint complete)
  [Transformer] SKIPPED (checkpoint complete)

── Fold hold-out=Monza  train=117,286  test=28,148  pos_rate=0.285 ──
  [CNN] SKIPPED (checkpoint complete)
  [LSTM] SKIPPED (checkpoint complete)
  [GRU] SKIPPED (checkpoint complete)
  [TCN] SKIPPED (checkpoint complete)
  [Transformer] SKIPPED (checkpoint complete)

── Fold hold-out=Silverstone  train=122,321  test=23,113  pos_rate=0.330 ──
  [CNN] SKIPPED (checkpoint complete)
  [LSTM] SKIPPED (checkpoint complete)
  [GRU] SKIPPED (checkpoint complete)
  [TCN] SKIPPED (checkpoint complete)
  [Transformer] training …


/tmp/ipykernel_58/1478295488.py:40: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp_scaler = torch.cuda.amp.GradScaler() if USE_AMP else None
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/

        ep=  5  loss=0.0099  val_AUC-PR=0.9928  best=0.9928  no_imp=0  lr=3.0e-04


/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


        ep= 10  loss=0.0090  val_AUC-PR=0.9942  best=0.9942  no_imp=0  lr=3.0e-04


...

        ep= 85  loss=0.0056  val_AUC-PR=0.9966  best=0.9967  no_imp=7  lr=9.4e-06


/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


        ep= 90  loss=0.0056  val_AUC-PR=0.9966  best=0.9967  no_imp=12  lr=4.7e-06


/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  [Transformer] DONE  ep= 93  F1=0.596  AUC-ROC=0.857  AUC-PR=0.685

deep_results.csv saved (6 rows).

Mean across LOCO folds (H=10):
             f1_macro  auc_roc  auc_pr
model                                 
GRU            0.8124   0.9165  0.8477
Transformer    0.7548   0.9090  0.7835
LSTM           0.5282   0.7649  0.5810
CNN            0.5936   0.6837  0.5617
TCN            0.6092   0.5374  0.4924


In [22]:
# Cell 11 — Horizon sweep: top-2 models, all H in [1,5,10,25,50]
# Auto-picks best 2 by AUC-ROC from Cell 10; override SWEEP_MODELS if desired.
SWEEP_MODELS = ["TCN", "Transformer"]   # fallback default

if deep_rows:
    ranking = (pd.DataFrame(deep_rows)
               .groupby("model")["auc_roc"].mean()
               .sort_values(ascending=False))
    print("Model ranking (H=10 mean AUC-ROC):")
    print(ranking.to_string())
    SWEEP_MODELS = list(ranking.index[:2])
    print(f"\nAuto-selected for sweep: {SWEEP_MODELS}")
else:
    print(f"No Cell 10 results — using default sweep models: {SWEEP_MODELS}")

sweep_rows = []

for H in HORIZONS:
    if H not in windows_cache:
        continue
    X, y, circs, _ = windows_cache[H]
    N, Ww, Ff = X.shape

    for held_out in circuit_list:
        te_m = circs == held_out
        tr_m = ~te_m
        n_tr = int(tr_m.sum())
        scaler = scalers_cache[H][held_out]

        tr_idx = np.where(tr_m)[0]
        try:
            idx_tr, idx_val = train_test_split(
                tr_idx, test_size=0.10, stratify=y[tr_m], random_state=SEED)
        except ValueError:
            idx_tr, idx_val = train_test_split(tr_idx, test_size=0.10, random_state=SEED)

        train_ds    = make_tensor_ds(X[idx_tr],  y[idx_tr],  scaler)
        val_ds      = make_tensor_ds(X[idx_val], y[idx_val], scaler)
        test_ds     = make_tensor_ds(X[te_m],    y[te_m],    scaler)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2,
                                 shuffle=False, num_workers=0)

        pos_rate  = float(y[idx_tr].mean())
        criterion = FocalLoss(alpha=1. - pos_rate, gamma=2.).to(DEVICE)

        for mname in SWEEP_MODELS:
            pk = progress_key(mname, H, held_out)
            if pk in completed:
                continue

            print(f"  [{mname}] H={H:2d}  fold={held_out} …", flush=True)
            try:
                set_seed(SEED)
                model = build_model(mname, W, F_DIM).to(DEVICE)
                model, ep = fit_fold(mname, model, train_ds, val_ds,
                                     criterion, held_out, H)

                y_true, y_prob, y_pred = evaluate_model(model, test_loader)
                m = compute_metrics(y_true, y_prob, y_pred)

                print(f"    ep={ep:3d}  F1={m['f1_macro']:.3f}  "
                      f"AUC-ROC={m['auc_roc']:.3f}  AUC-PR={m['auc_pr']:.3f}")

                sweep_rows.append({"model": mname, "held_out": held_out, "H": H,
                                   "n_train": n_tr, "stopped_epoch": ep, **m})

                torch.save({"model_state": model.state_dict(), "W": W, "F": F_DIM,
                            "H": H, "held_out": held_out,
                            "epoch": ep, "metrics": m},
                           MODELS_DIR / f"{mname}_fold_{held_out}_H{H:03d}.pt")

                completed.add(pk)
                save_progress(completed)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            except Exception:
                print(f"    [ERROR] {mname} H={H} fold={held_out}:"); traceback.print_exc()

if sweep_rows:
    sweep_df = pd.DataFrame(sweep_rows)
    sweep_df.to_csv(RESULTS_DIR / "sweep_results.csv", index=False)
    print(f"\nsweep_results.csv saved ({len(sweep_rows)} rows).")
    print("\nHorizon sweep — mean AUC-ROC (rows=model, cols=H):")
    print(sweep_df.groupby(["model","H"])["auc_roc"].mean()
          .unstack("H").round(4).to_string())


Model ranking (H=10 mean AUC-ROC):
model
GRU            0.916518
Transformer    0.908992
LSTM           0.764891
CNN            0.683679
TCN            0.537377

Auto-selected for sweep: ['GRU', 'Transformer']
  [GRU] H= 1  fold=Monaco …


/tmp/ipykernel_58/1478295488.py:40: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp_scaler = torch.cuda.amp.GradScaler() if USE_AMP else None
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/

        ep=  5  loss=0.0036  val_AUC-PR=0.9993  best=0.9993  no_imp=0  lr=3.0e-04


/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


        ep= 10  loss=0.0033  val_AUC-PR=0.9992  best=0.9993  no_imp=5  lr=3.0e-04


...

        ep= 80  loss=0.0099  val_AUC-PR=0.9911  best=0.9912  no_imp=6  lr=4.7e-06


/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


        ep= 85  loss=0.0098  val_AUC-PR=0.9911  best=0.9912  no_imp=11  lr=4.7e-06


/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1478295488.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


    ep= 89  F1=0.389  AUC-ROC=0.574  AUC-PR=0.354

sweep_results.csv saved (32 rows).

Horizon sweep — mean AUC-ROC (rows=model, cols=H):
H                1       5       25      50
model                                      
GRU          0.9994  0.9863  0.6821  0.5051
Transformer  0.9993  0.9870  0.6907  0.5945


In [23]:
# Cell 12 — Results summary & full comparison table
import warnings; warnings.filterwarnings("ignore")

def load_results_csv():
    out = {}
    for tag, fname in [("baseline","baseline_results.csv"),
                       ("deep","deep_results.csv"),
                       ("sweep","sweep_results.csv")]:
        p = RESULTS_DIR / fname
        if p.exists():
            out[tag] = pd.read_csv(p)
            print(f"Loaded {tag}: {len(out[tag])} rows")
        else:
            print(f"[MISS] {tag} — not yet generated")
    return out

res = load_results_csv()

# ── H=10 full model comparison ─────────────────────────────────────────────────
if "baseline" in res and "deep" in res:
    h10 = pd.concat([
        res["baseline"][res["baseline"]["H"] == 10],
        res["deep"],
    ], ignore_index=True)

    print("\n══ H=10 mean metrics across LOCO folds ══════════════════════")
    print(h10.groupby("model")[["f1_macro","f1_xmode","auc_roc","auc_pr"]]
          .mean().round(4).sort_values("auc_roc", ascending=False).to_string())

    print("\n══ Per-fold AUC-ROC heatmap (H=10) ══════════════════════════")
    pivot = h10.pivot_table(values="auc_roc", index="model",
                            columns="held_out", aggfunc="mean")
    print(pivot.round(4).to_string())

    print("\n══ RF-lag vs best deep model (key paper finding) ════════════")
    rf_lag = h10[h10["model"] == "RF-lag"][["held_out","auc_roc"]].set_index("held_out")
    best   = h10.groupby(["model","held_out"])["auc_roc"].mean().unstack("held_out")
    print("RF-lag AUC-ROC per circuit:")
    print(rf_lag.round(4).to_string())

if "sweep" in res:
    print("\n══ Horizon sweep (mean AUC-ROC, rows=model, cols=H) ═════════")
    print(res["sweep"].groupby(["model","H"])["auc_roc"].mean()
          .unstack("H").round(4).to_string())


Loaded baseline: 60 rows
Loaded deep: 6 rows
Loaded sweep: 32 rows

══ H=10 mean metrics across LOCO folds ══════════════════════
             f1_macro  f1_xmode  auc_roc  auc_pr
model                                           
LR-instant     0.9062    0.8882   0.9626  0.8811
RF-instant     0.8724    0.8208   0.9588  0.8640
RF-lag         0.7940    0.6871   0.9445  0.8154
GRU            0.8124    0.7401   0.9165  0.8477
Transformer    0.7548    0.6363   0.9090  0.7835
LSTM           0.5282    0.2556   0.7649  0.5810
CNN            0.5936    0.3762   0.6837  0.5617
TCN            0.6092    0.4089   0.5374  0.4924

══ Per-fold AUC-ROC heatmap (H=10) ══════════════════════════
held_out     Monaco   Monza  Silverstone  Suzuka
model                                           
CNN             NaN     NaN          NaN  0.6837
GRU             NaN     NaN          NaN  0.9165
LR-instant   0.9800  0.9317       0.9702  0.9685
LSTM            NaN     NaN          NaN  0.7649
RF-instant   0.9554  0.

In [24]:
# Cell 13 — Save all outputs + print CSV content (paste back to Claude)
import shutil

# Copy to /kaggle/working/ root so the "Output" tab shows them for download
for fname in ["baseline_results.csv","deep_results.csv","sweep_results.csv"]:
    src = RESULTS_DIR / fname
    dst = WORK_DIR / fname
    if src.exists():
        shutil.copy(src, dst)
        print(f"Output: {dst}")

print(f"\nModels saved: {len(list(MODELS_DIR.glob('*.pt')))} .pt files")
print(f"Progress    : {len(completed)} folds complete")

# ── Print CSV content inline ───────────────────────────────────────────────────
# Copy and paste everything between the === markers back to Claude.
for name in ["baseline_results","deep_results","sweep_results"]:
    path = WORK_DIR / f"{name}.csv"
    if path.exists():
        print(f"\n{'='*70}")
        print(f">>> PASTE TO CLAUDE: {name}.csv")
        print("="*70)
        print(path.read_text())

print("\nDone! Paste the CSV blocks above back to Claude to fill the paper.")


Output: /kaggle/working/baseline_results.csv
Output: /kaggle/working/deep_results.csv
Output: /kaggle/working/sweep_results.csv

Models saved: 52 .pt files
Progress    : 52 folds complete

>>> PASTE TO CLAUDE: baseline_results.csv
model,held_out,H,n_test,xmode_rate,f1_macro,f1_xmode,f1_zmode,auc_roc,auc_pr
LR-instant,Monaco,1,36381,0.13314642258321652,0.9871872612649377,0.977763923524522,0.9966105990053533,0.9996465426019592,0.997823049814899
RF-instant,Monaco,1,36381,0.13314642258321652,0.9861657715921079,0.9759900218272529,0.996341521356963,0.999756446538999,0.9984571128478418
RF-lag,Monaco,1,36381,0.13314642258321652,0.9818551215593557,0.9684717712370378,0.9952384718816737,0.9995839105480208,0.9973326198750114
LR-instant,Monza,1,28598,0.5685012937967691,0.9818834285081982,0.9846517835476826,0.9791150734687138,0.9990430991908634,0.9992507806345282
RF-instant,Monza,1,28598,0.5685012937967691,0.9901120579698455,0.9915251644485238,0.9886989514911673,0.9994239988172885,0.9995632515973466